# Build a custom filter with Dynestyx

A custom filter plugs into the same `dsx.Filter` handler as a built-in filter. You supply two callbacks. `build_filter(model)` binds an algorithm to the current model, and `gather_outputs(state)` selects what each observation contributes to the result. Dynestyx runs the time loop, constructs posterior distributions, and registers the likelihood when you call `dsx.sample`.

We'll implement the combined Cuthbert interface, expose the innovation as a custom diagnostic, and check the result against an exact Kalman filter. The final example compares the custom filter with Dynestyx's built-in UKF on a transition with an interaction between state coordinates.

In [ ]:
from typing import NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import numpyro.distributions as dist
from numpyro.handlers import seed, trace
from cuthbert.inference import Filter as CuthbertFilter
from cuthbertlib.quadrature import gauss_hermite

import dynestyx as dsx
from dynestyx.configs import CustomFilterConfig, KFConfig, UKFConfig
from dynestyx.inference.custom_filter import FilterOutputs
from dynestyx.inference.observation_predictions import PredictedObservationOutputs

## Decide what a step returns

The state is yours to design. This algorithm needs a filtered mean, covariance, and running log likelihood for the next step. It also carries the predictive observation moments and innovation so the gather callback can expose them. Keep the same field shapes at every step for JAX's scan.

In [ ]:
class State(NamedTuple):
    mean: jax.Array
    cov: jax.Array
    loglik: jax.Array
    predictions: PredictedObservationOutputs
    innovation: jax.Array

In [ ]:
def gather_outputs(state):
    return FilterOutputs(
        log_normalizing_constant=state.loglik,
        filtered_means=state.mean,
        filtered_covariances=state.cov,
        predicted_observations=state.predictions,
        extra_outputs={"innovation": state.innovation},
    )

`FilterOutputs` is the boundary between your algorithm and Dynestyx. The Gaussian fields become one distribution per observation in `result.dists`. The last cumulative log likelihood becomes `result.marginal_loglik`, and named extras are stacked along observation time. Here the innovation is $y_k-\hat y_{k|k-1}$, so saving it also gives us a way to check that predictions were gathered before assimilation.

## Bind the callbacks to a model

The factory receives the `DynamicalModel` for this invocation, including its current parameters. `init_prepare` creates the prior, `filter_prepare` packages one observation without needing the previous state, and `filter_combine` assimilates it. At the first observation, update the initial prior directly. At later observations, use the previous control for the transition and the current control for the observation. The two numerical helpers below are the parts you would replace for a different Gaussian algorithm.

If your algorithm already has separate prediction and update functions, return a `SplitFilter(initialize, predict, update)` from the factory. Its gather callback receives `(prior, filtered)`. The config and the calls to `dsx.condition` and `dsx.sample` are the same.

In [ ]:
def build_filter(model):
    rule = gauss_hermite.weights(model.state_dim, order=3)

    def initialize(*, key):
        prior = model.initial_condition
        zero = jnp.zeros(model.observation_dim, dtype=prior.mean.dtype)
        zeros = jnp.outer(zero, zero)
        predictions = PredictedObservationOutputs(mean=zero, cov=zeros, obs_cov=zeros)
        return State(prior.mean, prior.covariance_matrix, jnp.zeros_like(zero[0]),
                     predictions, zero)

    def prepare(inputs, *, key):
        return inputs

    def combine(state, inputs):
        def predict():
            transition = lambda x: model.state_evolution(
                x, inputs.u_prev, inputs.time_prev, inputs.time
            )
            mean, signal, noise, _ = transform(rule, state.mean, state.cov, transition)
            return mean, signal + noise

        mean, cov = jax.lax.cond(
            inputs.is_first_step, lambda: (state.mean, state.cov), predict
        )
        observe = lambda x: model.observation_model(x, inputs.u, inputs.time)
        return assimilate(rule, mean, cov, observe, inputs.y, state.loglik)

    return CuthbertFilter(init_prepare=initialize, filter_prepare=prepare,
                          filter_combine=combine, associative=False)


config = CustomFilterConfig(build_filter=build_filter, gather_outputs=gather_outputs)

## Supply the numerical step

This example uses order-three Gauss–Hermite quadrature. For a Gaussian input, `transform` evaluates the model at weighted points and returns four quantities: the output mean, the covariance of the signal, the average noise covariance, and the state–signal cross-covariance. `assimilate` uses them for a Gaussian update and adds the predictive log density of this observation to the running total.

In [ ]:
def transform(rule, mean, cov, conditional):
    points = rule.get_sigma_points(mean, jnp.linalg.cholesky(cov))

    def moments(x):
        distribution = conditional(x)
        return distribution.mean, distribution.covariance_matrix

    means, covs = jax.vmap(moments)(points.points)
    signals = points._replace(points=means)
    noise = jnp.einsum("i,ijk->jk", points.wm, covs)
    return signals.mean, signals.covariance(), noise, points.covariance(signals)


def assimilate(rule, mean, cov, observe, y, previous_loglik):
    y_mean, signal, noise, cross = transform(rule, mean, cov, observe)
    y_cov = signal + noise
    gain = jnp.linalg.solve(y_cov, cross.T).T
    innovation = y - y_mean
    return State(
        mean=mean + gain @ innovation,
        cov=cov - gain @ cross.T,
        loglik=previous_loglik + dist.MultivariateNormal(y_mean, y_cov).log_prob(y),
        predictions=PredictedObservationOutputs(mean=y_mean, cov=signal, obs_cov=y_cov),
        innovation=innovation,
    )

## Use the config with conditioning and NumPyro

Consider two independent standard-normal coordinates. We observe the first with noise variance $0.5$. In the transition, the product of both coordinates becomes the next first coordinate, with process noise variance $0.1$. Both filters below receive the same model and the observations 0 and 1.

In [ ]:
q, r = 0.1, 0.5
model = dsx.DynamicalModel(
    control_dim=0,
    initial_condition=dist.MultivariateNormal(jnp.zeros(2), jnp.eye(2)),
    state_evolution=dsx.GaussianStateEvolution(
        F=lambda x, u, t, t_next: jnp.array([x[0] * x[1], x[1]]),
        cov=q * jnp.eye(2),
    ),
    observation_model=dsx.LinearGaussianObservation(H=jnp.array([[1., 0.]]),
                                                    R=r * jnp.eye(1)),
)
data = dict(obs_times=jnp.array([0., 1.]), obs_values=jnp.array([[0.], [1.]]))
with dsx.Filter(config):
    result = dsx.condition("x", model, **data)

np.testing.assert_allclose(result.extra_outputs["innovation"],
                           data["obs_values"] - result.predicted_observations.mean)
print("Innovation at each observation:", result.extra_outputs["innovation"].ravel())
print("Filtered mean after the second observation:", result.dists[-1].mean)

In [ ]:
with trace() as sites, seed(rng_seed=0), dsx.Filter(config):
    dsx.sample("x", model, **data)

factor = sites["x_marginal_log_likelihood"]["fn"].log_factor
np.testing.assert_allclose(factor, result.marginal_loglik, atol=1e-6)
print("Log likelihood registered with NumPyro:", factor)

## Check the callback wiring on a linear model

The same factory can bind to a different model. On linear dynamics and observations, the quadrature moments are exact. Comparing every posterior and every cumulative likelihood with the Kalman filter checks initialization, time alignment, and the output mapping.

In [ ]:
linear = dsx.LTI_discrete(A=jnp.array([[0.8]]), Q=0.1*jnp.eye(1),
                          H=jnp.eye(1), R=0.2*jnp.eye(1))
with dsx.Filter(config):
    gh = dsx.condition("x", linear, **data)
with dsx.Filter(KFConfig(filter_source="cuthbert")):
    exact = dsx.condition("x", linear, **data)
for a, b in zip(gh.dists, exact.dists, strict=True):
    np.testing.assert_allclose(a.mean, b.mean, atol=1e-5)
    np.testing.assert_allclose(a.covariance_matrix, b.covariance_matrix, atol=1e-5)
np.testing.assert_allclose(gh.log_normalizing_constant, exact.log_normalizing_constant, atol=1e-5)
np.testing.assert_allclose(gh.marginal_loglik, exact.marginal_loglik, atol=1e-5)

## Compare with the built-in UKF

After the first observation, the two coordinates are independent with means zero and variances $v=r/(1+r)$ and $1$. Their product has variance $v$, so the exact variance of the second **observation** is $v+q+r$. The UKF's five sigma points in two dimensions lie on the coordinate axes of this diagonal Gaussian. Every point therefore has product zero. The custom rule evaluates nine points, including off-axis points that capture the missing variance.

We can also check the final **filtered mean** against a numerical Bayesian reference. Conditional on the previous second coordinate $b$, the next first coordinate is Gaussian with variance $s_b^2=v b^2+q$. For the second observation $y$, its conditional posterior mean is $y s_b^2/(s_b^2+r)$. Average this over $b$, weighting its standard-normal prior by the observation likelihood $mathcal N(y;0,s_b^2+r)$.

In [ ]:
from scipy.integrate import quad
from scipy.stats import norm

with dsx.Filter(UKFConfig()):
    ukf = dsx.condition("x", model, **data)

v = r / (1 + r)
y = float(data["obs_values"][-1, 0])

def state_variance(b):
    return v * b**2 + q

def weight(b):
    return norm.pdf(b) * norm.pdf(y, scale=np.sqrt(state_variance(b) + r))

def weighted_mean(b):
    variance = state_variance(b)
    return weight(b) * y * variance / (variance + r)

evidence = quad(weight, -np.inf, np.inf)[0]
reference_mean = quad(weighted_mean, -np.inf, np.inf)[0] / evidence
reference_var = v + q + r

# The discrete UKF stores the state forecast from observation 0 to observation 1.
# H selects coordinate 0, so adding R gives its predictive observation variance.
rows = [
    ("Bayesian reference", reference_var, reference_mean),
    ("Custom filter", float(result.predicted_observations.obs_cov[1, 0, 0]),
     float(result.dists[1].mean[0])),
    ("Built-in UKF", float(ukf.states.predicted_covariances[0, 0, 0]) + r,
     float(ukf.dists[1].mean[0])),
]
print(f"{'Method':<20} {'Observation variance':>22} {'Filtered mean':>16} {'Abs. mean error':>17}")
for name, variance, mean in rows:
    print(f"{name:<20} {variance:>22.4f} {mean:>16.4f} {abs(mean-reference_mean):>17.4f}")
np.testing.assert_allclose(rows[1][1], reference_var, atol=1e-5)

The custom filter gives the exact predictive observation variance of **0.9333**. The UKF gives **0.6000**, so it gives the second observation less weight. Against the numerical Bayesian reference, the absolute error in the filtered mean is **0.0831** for the custom filter and **0.2145** for the UKF. The remaining error comes from the Gaussian approximation to the product distribution. The custom rule uses $3^d$ model evaluations per transform, compared with $2d+1$ for the UKF.